# FRC Bumper Detection Model Training

在 Google Colab T4 GPU 上訓練 YOLOv26n bumper 偵測模型。

**訓練流程:**
1. 本地執行 `merge_datasets.py` 產生 `datasets/merged.zip`（1826 張，人工審核過）
2. 上傳 zip 到 Google Drive
3. 訓練 → 匯出 ONNX → 下載

**產出**: `frc_robot.onnx` — 下載後放到 `models/` 目錄即可

## 0. 確認 GPU 可用

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("Please go to Runtime → Change runtime type → T4 GPU")

## 1. 安裝套件

In [ ]:
!pip install -q ultralytics onnx onnxruntime

## 2. 上傳資料集

**方法：** 先把 `datasets/merged.zip` 上傳到 Google Drive，再從 Drive 解壓。

1. 開 [drive.google.com](https://drive.google.com/)
2. 把 `datasets/merged.zip`（398 MB）拖進 `Colab Notebooks/` 資料夾
3. 執行下方儲存格

In [ ]:
import zipfile
from google.colab import drive

# 掛載 Google Drive
drive.mount("/content/drive")

# 解壓資料集
zip_path = "/content/drive/MyDrive/Colab Notebooks/merged.zip"
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall("/content/dataset")

# 修正 data.yaml 路徑 (YOLO 需要絕對路徑)
import pathlib
p = pathlib.Path("/content/dataset/merged/data.yaml")
p.write_text(p.read_text().replace(
    p.read_text().split("path:")[1].split("\n")[0],
    " /content/dataset/merged"
))

print(f"資料集已解壓至: /content/dataset/merged")
!ls /content/dataset/merged/
!echo "---"
!echo "Train:" && ls /content/dataset/merged/images/train/ | wc -l
!echo "Val:" && ls /content/dataset/merged/images/val/ | wc -l

## 3. 訓練模型

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/content/dataset/merged/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device="cuda:0",
    project="/content/runs",
    name="frc_robot",
    exist_ok=True,
    verbose=True,
)

print(f"\n訓練完成！最佳模型: {model.trainer.best}")

## 4. 驗證模型

In [ ]:
metrics = model.val(data="/content/dataset/merged/data.yaml")
print(f"\n驗證結果:")
print(f"  mAP50:     {metrics.box.map50:.3f}")
print(f"  mAP50-95:  {metrics.box.map:.3f}")
print(f"  Precision: {metrics.box.mp:.3f}")
print(f"  Recall:    {metrics.box.mr:.3f}")

## 5. 匯出 ONNX

In [ ]:
import shutil
from pathlib import Path

export_path = model.export(format="onnx", imgsz=640, simplify=True)
print(f"ONNX 匯出至: {export_path}")

output = Path("/content/frc_robot.onnx")
shutil.copy2(export_path, output)

size_mb = output.stat().st_size / (1024 * 1024)
print(f"模型大小: {size_mb:.1f} MB")
print(f"\n模型已準備好下載: {output}")

## 6. 下載模型

執行下方儲存格，瀏覽器會自動下載 `frc_robot.onnx`。

下載後放到專案的 `models/frc_robot.onnx` 即可。

In [ ]:
from google.colab import files
files.download("/content/frc_robot.onnx")
print("下載完成後，將 frc_robot.onnx 放到專案的 models/ 目錄")

## 7. (選用) 視覺化偵測結果

In [ ]:
import glob
from IPython.display import Image, display

results = model.predict(
    source="/content/dataset/merged/images/val",
    save=True,
    project="/content/runs",
    name="predictions",
    exist_ok=True,
    conf=0.25,
)

pred_images = sorted(glob.glob("/content/runs/predictions/*.jpg"))[:6]
for img_path in pred_images:
    display(Image(filename=img_path, width=600))
    print()